# Session Memory Compaction (TypeScript)

Long-running conversations with Claude can exceed context limits, causing loss of important information. Whether you're building a coding assistant, creative writing tool, or customer service agent, managing session memory is critical for maintaining continuity and quality.

This cookbook teaches you how to **proactively manage session memory** to avoid jarring context limit interruptions. Unlike reactive approaches that wait until the context is full, you'll learn to build session memory in the background so compaction is instant when needed.

**Related:** For automatic SDK-based compaction in agentic workflows, see [Automatic Context Compaction](../tool_use/automatic-context-compaction.ipynb). This cookbook focuses on manual control patterns for conversational applications.

## Learning Objectives

By the end of this cookbook, you will be able to:

- Write effective session memory prompts that preserve critical context across compaction events
- Implement **instant compaction** using background Promises to eliminate user wait time
- Apply prompt caching to reduce the cost of background memory updates by ~80%
- Choose appropriate compaction strategies (traditional vs. instant) based on your use case

## Prerequisites and Setup

This notebook uses the **Deno kernel** for TypeScript. Deno handles package imports natively — no `npm install` needed.

Set your `ANTHROPIC_API_KEY` in a `.env` file at the project root:
```
ANTHROPIC_API_KEY=sk-ant-...
```

In [ ]:
import Anthropic from 'npm:@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

await load({ envPath: '../.env', export: true });

const apiKey = Deno.env.get('ANTHROPIC_API_KEY');
if (!apiKey) throw new Error('ANTHROPIC_API_KEY not found in .env');

const client = new Anthropic({ apiKey });
const MODEL = 'claude-sonnet-4-6';
console.log('Anthropic client initialized. Model:', MODEL);

In [ ]:
// ── Types ──────────────────────────────────────────────────────────────────
type Role = 'user' | 'assistant';
interface SimpleMessage { role: Role; content: string; }

// ── Helper functions ────────────────────────────────────────────────────────
function truncateResponse(text: string, maxLines = 15): string {
  const lines = text.trim().split('\n');
  if (lines.length <= maxLines) return text;
  return lines.slice(0, maxLines).join('\n') + `\n... (${lines.length - maxLines} more lines)`;
}

function removeThinkingBlocks(text: string): [string, string] {
  const matches = [...text.matchAll(/<think>[\s\S]*?<\/think>/g)].map(m => m[0]);
  const cleaned = text.replace(/<think>[\s\S]*?<\/think>\s*/g, '').trim();
  return [cleaned, matches.join('')];
}

/**
 * Add cache_control to the last user message for prompt caching.
 * Normalises all messages to list-of-blocks format so the prefix is
 * identical across the main chat and the background summariser calls.
 */
function addCacheControl(messages: SimpleMessage[]): Anthropic.MessageParam[] {
  let lastUserIdx = -1;
  for (let i = 0; i < messages.length; i++) {
    if (messages[i].role === 'user') lastUserIdx = i;
  }

  return messages.map((msg, i) => {
    const block: Record<string, unknown> = { type: 'text', text: msg.content };
    if (i === lastUserIdx) block.cache_control = { type: 'ephemeral' };
    return { role: msg.role, content: [block] } as Anthropic.MessageParam;
  });
}

function estimateTokens(text: string): number {
  return Math.floor(text.length / 4);
}

console.log('Helper functions defined.');

In [ ]:
const SESSION_MEMORY_PROMPT = `
Compress the conversation into a structured summary
that preserves all information needed to continue work seamlessly. Optimize for the assistant's
ability to continue working, not human readability.

<analysis-instructions>
Before generating your summary, analyze the transcript in <think>...</think> tags:
1. What did the user originally request? (Exact phrasing)
2. What actions succeeded? What failed and why?
3. Did the user correct or redirect the assistant at any point?
4. What was actively being worked on at the end?
5. What tasks remain incomplete or pending?
6. What specific details (IDs, paths, values, names) must survive compression?
</analysis-instructions>

<summary-format>
## User Intent
The user's original request and any refinements. Use direct quotes for key requirements.
If the user's goal evolved during the conversation, capture that progression.

## Completed Work
Actions successfully performed. Be specific:
- What was created, modified, or deleted
- Exact identifiers (file paths, record IDs, URLs, names)
- Specific values, configurations, or settings applied

## Errors & Corrections
- Problems encountered and how they were resolved
- Approaches that failed (so they aren't retried)
- User corrections: "don't do X", "actually I meant Y", "that's wrong because..."
Capture corrections verbatim — these represent learned preferences.

## Active Work
What was in progress when the session ended. Include:
- The specific task being performed
- Direct quotes showing exactly where work left off
- Any partial results or intermediate state

## Pending Tasks
Remaining items the user requested that haven't been started.
Distinguish between "explicitly requested" and "implied/assumed."

## Key References
Important details needed to continue:
- Identifiers: IDs, paths, URLs, names, keys
- Values: numbers, dates, configurations, credentials (redacted)
- Context: relevant background information, constraints, preferences
- Citations: sources referenced during the conversation
</summary-format>

<preserve-rules>
Always preserve when present:
- Exact identifiers (IDs, paths, URLs, keys, names)
- Error messages verbatim
- User corrections and negative feedback
- Specific values, formulas, or configurations
- Technical constraints or requirements discovered
- The precise state of any in-progress work
</preserve-rules>

<compression-rules>
- Weight recent messages more heavily — the end of the transcript is the active context
- Omit pleasantries, acknowledgments, and filler ("Sure!", "Great question")
- Omit system context that will be re-injected separately
- Keep each section under 500 words; condense older content to make room for recent
- If you must cut details, preserve: user corrections > errors > active work > completed work
</compression-rules>
`;

console.log('SESSION_MEMORY_PROMPT defined.');

### Traditional Compaction

In **traditional compaction**, a summary is generated on-demand once the token threshold is reached.
The user must wait for the summary before their next message is processed.

```
TRADITIONAL COMPACTION (slow)
─────────────────────────────
Turn 1 → Turn 2 → Turn 3 → ... → Turn N → CONTEXT FULL!
                                              │
                                              ▼
                                    ┌─────────────────┐
                                    │ Generate summary│
                                    │ ( USER WAITS !) │
                                    └─────────────────┘
                                              │
                                              ▼
                                         Continue
```

In [ ]:
class TraditionalCompactingChatSession {
  private messages: SimpleMessage[] = [];
  private currentContextWindowTokens = 0;
  private summary: string | null = null;

  constructor(
    private systemMessage = 'You are a helpful assistant',
    private contextLimit = 10_000,
  ) {}

  async chat(userMessage: string): Promise<[string, Anthropic.Usage]> {
    // In traditional compaction we check AFTER the user sends a message — not ideal!
    if (this.currentContextWindowTokens >= this.contextLimit) {
      console.log(
        `\n🧹 Context window at ${this.currentContextWindowTokens} tokens. Limit exceeded, compacting...`,
      );
      await this.compact();
    }

    this.messages.push({ role: 'user', content: userMessage });

    const response = await client.messages.create({
      model: MODEL,
      max_tokens: 3500,
      system: this.systemMessage,
      messages: addCacheControl(this.messages),
    });

    const assistantText =
      response.content[0].type === 'text' ? response.content[0].text : '';
    this.messages.push({ role: 'assistant', content: assistantText });

    const cacheRead = (response.usage as Record<string, number>).cache_read_input_tokens ?? 0;
    const totalInput = response.usage.input_tokens + cacheRead;
    this.currentContextWindowTokens = totalInput + response.usage.output_tokens;

    console.log(
      `Input=${totalInput.toLocaleString()}, Cache hit=${cacheRead > 0} | ` +
        `Output=${response.usage.output_tokens.toLocaleString()} | ` +
        `Messages=${this.messages.length}`,
    );
    return [assistantText, response.usage];
  }

  private async compact(): Promise<void> {
    const start = performance.now();

    const compactionMessages: Anthropic.MessageParam[] = [
      ...addCacheControl(this.messages),
      { role: 'user', content: SESSION_MEMORY_PROMPT },
    ];

    const response = await client.messages.create({
      model: MODEL,
      max_tokens: 5000,
      system: this.systemMessage,
      messages: compactionMessages,
    });

    const elapsed = (performance.now() - start) / 1000;
    const rawSummary = response.content[0].type === 'text' ? response.content[0].text : '';
    const [cleanedSummary, removedThinking] = removeThinkingBlocks(rawSummary);
    this.summary = cleanedSummary;

    const approxSummaryTokens =
      response.usage.output_tokens - Math.round(removedThinking.length / 4);
    const reduction = this.currentContextWindowTokens - approxSummaryTokens;
    const pct = ((reduction / this.currentContextWindowTokens) * 100).toFixed(0);
    const cacheRead = (response.usage as Record<string, number>).cache_read_input_tokens ?? 0;

    this.messages = [
      {
        role: 'user',
        content: `This session is being continued from a previous conversation. Here is the session memory: ${this.summary}. Continue from where we left off.`,
      },
    ];

    console.log(`\n${'-'.repeat(60)}`);
    console.log('📝 New session memory created.');
    console.log(
      `✅ Tokens reduced: ${this.currentContextWindowTokens.toLocaleString()} → ${approxSummaryTokens} (${reduction.toLocaleString()} saved, ${pct}% reduction)`,
    );
    console.log(`⏱️  Compaction time: ${elapsed.toFixed(2)}s (user waiting...)`);
    console.log(`   Cache used: ${cacheRead > 0}`);
    console.log(`${'-'.repeat(60)}`);

    this.currentContextWindowTokens = approxSummaryTokens;
  }
}

console.log('TraditionalCompactingChatSession defined.');

Below we simulate a conversation between an author and an LLM that helps write stories.

In [ ]:
const SYSTEM_PROMPT = `
You are a short story writer who helps authors develop their ideas into compelling narratives.

## What You Do

**Plot Development**
- Help authors work through story structure, pacing, and narrative arc
- Identify plot holes, inconsistencies, or missed opportunities
- Suggest ways to raise stakes, add tension, or deepen conflict
- Brainstorm twists, resolutions, and scene transitions

**Character Development**
- Develop backstories, motivations, and internal conflicts
- Ensure characters have distinct voices and consistent behavior
- Explore character relationships and how they drive the plot
- Help authors understand what their characters want vs. what they need

**Drafting**
- Write short stories or scenes based on the author's ideas and direction
- Match tone, genre conventions, and stylistic preferences
- Show rather than tell when bringing scenes to life
- Craft dialogue that reveals character and advances plot

## How You Work
- You are the lead writer. When you disagree with a creative choice, say so respectfully, but ultimately defer to what the author wants.
- DO NOT ask the user to provide more context or clarify their request. Assume you have enough information to proceed.
`;

const session = new TraditionalCompactingChatSession(SYSTEM_PROMPT);

const demoMessages = [
  'I want to create a story about a young detective solving a mysterious case in a small town. Generate 3 well thought out plot ideas for me to consider.',
  "I don't like those ideas, can you think of one plot something more unique and unexpected?",
  'Ok I like it. Can you help me develop the main character\'s backstory and motivations?',
  'Can you draft a detailed outline for the story, breaking it down into chapters and key events?',
  'Can you draft me a first chapter based on the plot and character ideas we\'ve discussed so far? Make it around 2,000 words.',
  'Can you draft a second chapter that builds on the first one, introducing a new twist in the mystery?',
];

console.log('Starting conversation...\n');

let turnCount = 0;
for (const message of demoMessages) {
  turnCount++;
  console.log(`==============================================\nTurn ${turnCount}:`);
  console.log(`\nUser: ${message}`);
  const [reply] = await session.chat(message);
  console.log(`\nAssistant:\n${truncateResponse(reply, 10)}`);
}

On the next turn we will hit the context limit, which triggers compaction:

In [ ]:
const [titleReply] = await session.chat('Propose a title for the book');
console.log(`\nAssistant:\n${truncateResponse(titleReply, 10)}`);

You'll notice compaction blocked the response for several seconds — the user had to wait.

## Instant Compaction

With **instant compaction**, the session memory is **proactively** generated once a soft token threshold is reached, using a background Promise. When the hard limit is hit, the summary is already ready — zero wait time.

```
SESSION MEMORY COMPACTION (instant)
────────────────────────────────────
Turn 1 → Turn 2 → ... → Turn K → Turn K+1 → ... → Turn N → CONTEXT FULL!
                            │                                      │
              (soft threshold met:                                  │
             fire background Promise)                               │
                            │                                      │
                            ▼                                      │
                       ┌────────┐      (update on next turn)       │
                       │ Create │ ──────────────────────────►      │
                       │ memory │       ┌────────┐                 │
                       └────────┘       │ Update │                 │
                            │           │ memory │                 │
                            ▼           └────────┘                 ▼
                    📝 session-memory ──────────────────► INSTANT SWAP!
                      (runs in background,                (no wait time)
                      never blocks user)
```

In TypeScript we use a **fire-and-forget Promise** (without `await`) instead of Python threads.
The Promise runs concurrently on the same event loop and updates shared state when complete.

In [ ]:
class InstantCompactingChatSession {
  private messages: SimpleMessage[] = [];
  private currentContextWindowTokens = 0;
  private sessionMemory: string | null = null;
  private lastSummarizedIndex = 0;
  private tokensAtLastUpdate = 0;

  // Background Promise — null when idle, resolves when update completes
  private updatePromise: Promise<void> | null = null;

  constructor(
    private systemMessage = 'You are a helpful assistant',
    private contextLimit = 12_000,
    private minTokensToInit = 7_500,
    private minTokensBetweenUpdates = 2_000,
  ) {}

  async chat(userMessage: string): Promise<[string, Anthropic.Usage, string | null]> {
    if (this.currentContextWindowTokens + estimateTokens(userMessage) >= this.contextLimit) {
      await this.compact();
    }

    this.messages.push({ role: 'user', content: userMessage });

    const response = await client.messages.create({
      model: MODEL,
      max_tokens: 3500,
      system: this.systemMessage,
      messages: addCacheControl(this.messages),
    });

    const assistantText =
      response.content[0].type === 'text' ? response.content[0].text : '';
    this.messages.push({ role: 'assistant', content: assistantText });

    const cacheRead = (response.usage as Record<string, number>).cache_read_input_tokens ?? 0;
    const totalInput = response.usage.input_tokens + cacheRead;
    this.currentContextWindowTokens = totalInput + response.usage.output_tokens;

    // KEY: fire background update without awaiting — never blocks the user
    let backgroundStatus: string | null = null;
    if (this.shouldInitMemory() || this.shouldUpdateMemory()) {
      this.triggerBackgroundUpdate();
      backgroundStatus = this.sessionMemory === null ? 'initializing' : 'updating';
    }

    return [assistantText, response.usage, backgroundStatus];
  }

  private shouldInitMemory(): boolean {
    return (
      this.sessionMemory === null &&
      this.currentContextWindowTokens >= this.minTokensToInit
    );
  }

  private shouldUpdateMemory(): boolean {
    if (this.sessionMemory === null) return false;
    return (
      this.currentContextWindowTokens - this.tokensAtLastUpdate >= this.minTokensBetweenUpdates
    );
  }

  private triggerBackgroundUpdate(): void {
    if (this.updatePromise !== null) return; // already running

    const messagesSnapshot = [...this.messages];
    const snapshotIndex = messagesSnapshot.length;
    const currentTokens = this.currentContextWindowTokens;

    // Fire and forget — the Promise resolves in the background
    this.updatePromise = this.backgroundMemoryUpdate(messagesSnapshot, snapshotIndex, currentTokens)
      .catch(err => console.error('[Background] Error updating memory:', err))
      .finally(() => { this.updatePromise = null; });
  }

  private async backgroundMemoryUpdate(
    messagesSnapshot: SimpleMessage[],
    snapshotIndex: number,
    currentTokens: number,
  ): Promise<void> {
    let newMemory: string;

    if (this.sessionMemory === null) {
      newMemory = await this.createSessionMemory(messagesSnapshot);
    } else {
      const newMessages = messagesSnapshot.slice(this.lastSummarizedIndex);
      if (newMessages.length === 0) return;
      newMemory = await this.updateSessionMemory(newMessages);
    }

    // Update shared state after async work completes
    this.sessionMemory = newMemory;
    this.lastSummarizedIndex = snapshotIndex;
    this.tokensAtLastUpdate = currentTokens;
  }

  private async createSessionMemory(messages: SimpleMessage[]): Promise<string> {
    const compactionMessages: Anthropic.MessageParam[] = [
      ...addCacheControl(messages),
      { role: 'user', content: SESSION_MEMORY_PROMPT },
    ];
    const response = await client.messages.create({
      model: MODEL,
      max_tokens: 5000,
      system: this.systemMessage,
      messages: compactionMessages,
    });
    const raw = response.content[0].type === 'text' ? response.content[0].text : '';
    const [summary] = removeThinkingBlocks(raw);
    const cacheRead = (response.usage as Record<string, number>).cache_read_input_tokens ?? 0;
    console.log(`   [Background] Initial session memory created. Cache hit=${cacheRead > 0}`);
    return summary;
  }

  private async updateSessionMemory(newMessages: SimpleMessage[]): Promise<string> {
    const updatePrompt =
      SESSION_MEMORY_PROMPT +
      `\nThere is an existing session memory: ${this.sessionMemory}. Return the entire session memory with updates to reflect the new messages.`;
    const response = await client.messages.create({
      model: MODEL,
      max_tokens: 5000,
      system: this.systemMessage,
      messages: [
        ...addCacheControl(newMessages),
        { role: 'user', content: updatePrompt },
      ],
    });
    const raw = response.content[0].type === 'text' ? response.content[0].text : '';
    const [summary] = removeThinkingBlocks(raw);
    console.log('   [Background] Session memory updated.');
    return summary;
  }

  async compact(): Promise<void> {
    const prevCount = this.messages.length;

    // If background update is still running, wait for it
    if (this.sessionMemory === null) {
      if (this.updatePromise !== null) {
        console.log('   ⏳ Waiting for background memory update...');
        await this.updatePromise;
      }
      if (this.sessionMemory === null) {
        console.log('   ⚠️  No pre-built memory, creating synchronously...');
        const start = performance.now();
        this.sessionMemory = await this.createSessionMemory(this.messages);
        console.log(`   ⏱️  Took ${((performance.now() - start) / 1000).toFixed(2)}s (should be instant normally!)`);
        this.lastSummarizedIndex = this.messages.length;
      }
    }

    const unsummarized = this.messages.slice(this.lastSummarizedIndex);
    const summaryMessage: SimpleMessage = {
      role: 'user',
      content: `This session is being continued from a previous conversation. Here is the session memory: ${this.sessionMemory}. Continue from where we left off.`,
    };
    this.messages = [summaryMessage, ...unsummarized];
    this.lastSummarizedIndex = 1;

    console.log(`\n${'='.repeat(60)}`);
    console.log(`⚡ INSTANT COMPACTION! Messages: ${prevCount} → ${this.messages.length}`);
    console.log('   Session memory was pre-built (no wait time!)');
    console.log(`${'='.repeat(60)}`);
  }

  get memoryReady(): boolean { return this.sessionMemory !== null; }
  get messageCount(): number { return this.messages.length; }
}

console.log('InstantCompactingChatSession defined.');

### Example: Instant Compaction in Action

In [ ]:
const instantSession = new InstantCompactingChatSession(SYSTEM_PROMPT);

const instantMessages = [
  'I want to create a story about a young detective solving a mysterious case in a small town. Generate 3 well thought out plot ideas for me to consider.',
  "I don't like those ideas, can you think of one plot something more unique and unexpected?",
  'Ok I like it. Can you help me develop the main character\'s backstory and motivations?',
  'Can you draft a detailed outline for the story, breaking it down into chapters and key events?',
  'Can you draft me a first chapter based on the plot and character ideas we\'ve discussed so far? Make it around 2,000 words.',
  'Can you draft a second chapter that builds on the first one?',
];

console.log('Starting conversation with instant compacting chat session...\n');

let turn = 0;
for (const message of instantMessages) {
  const [reply, usage, bgStatus] = await instantSession.chat(message);
  turn++;

  const cacheRead = (usage as Record<string, number>).cache_read_input_tokens ?? 0;
  const totalInput = usage.input_tokens + cacheRead;

  console.log(`${'='.repeat(60)}`);
  console.log(`Turn ${turn}:\nUser: ${message}`);
  console.log(`\nAssistant:\n${truncateResponse(reply, 3)}`);
  console.log('\nToken Usage:');
  console.log(`  Input: ${totalInput.toLocaleString()} (new: ${usage.input_tokens.toLocaleString()}, cached: ${cacheRead.toLocaleString()})`);
  console.log(`  Output: ${usage.output_tokens.toLocaleString()}`);
  console.log(`  Messages: ${instantSession.messageCount} | Memory: ${instantSession.memoryReady ? 'ready' : 'not yet'}`);

  if (cacheRead > 0) {
    const cachePct = ((cacheRead / totalInput) * 100).toFixed(0);
    console.log(`  ✓ Cache hit! ${cachePct}% of input from cache`);
  }
  if (bgStatus) {
    console.log(`\n  [Background] Proactively ${bgStatus} session memory...`);
  }
  console.log();
}

In [ ]:
// This message pushes past the context limit — compaction should be instant!
const [finalReply, finalUsage] = await instantSession.chat('What did we just talk about? Give me one sentence');

const cacheRead = (finalUsage as Record<string, number>).cache_read_input_tokens ?? 0;
const totalInput = finalUsage.input_tokens + cacheRead;

console.log(`\nUser: What did we just talk about? Give me one sentence`);
console.log(`\nAssistant:\n${truncateResponse(finalReply, 3)}`);
console.log('\nToken Usage:');
console.log(`  Input: ${totalInput.toLocaleString()} (new: ${finalUsage.input_tokens.toLocaleString()}, cached: ${cacheRead.toLocaleString()})`);
console.log(`  Output: ${finalUsage.output_tokens.toLocaleString()}`);
console.log(`  Messages: ${instantSession.messageCount} | Memory: ${instantSession.memoryReady ? 'ready' : 'not yet'}`);

## Advanced: Understanding Prompt Caching

Background updates can be made **~80% cheaper** using prompt caching. The background summariser
reuses the same conversation prefix that was just sent to the main chat — so subsequent calls get
an automatic cache hit, and only the new summarisation instruction is billed at full price.

```
┌─────────────────────────────────────────────────────────────────────┐
│             COMPACTION + CACHING: Double benefit                    │
│                                                                     │
│  Main Chat                     Background Summariser               │
│  ─────────                     ─────────────────────               │
│                                                                     │
│  [Conversation grows...]       [Same conversation prefix]◆          │
│         │                               │ + [Summarise!]           │
│         │                               │                          │
│         │                    Cache hit! Only pays for              │
│         │                    the summarisation prompt              │
│         │                               │                          │
│         ▼                               ▼                          │
│  Context limit reached ────► Session memory ready instantly        │
│                               (built cheaply in background)        │
│                                                                     │
│  ◆ = cache_control breakpoint                                       │
└─────────────────────────────────────────────────────────────────────┘
```

| Scenario | Cost per background update | Notes |
|----------|---------------------------|-------|
| No caching | Full input cost | 5,000 tokens × $3/M = $0.015 |
| With caching | ~10% of input cost | 500 new + 4,500 cached = $0.003 |
| **Savings** | **~80%** | Compounds over many updates |

The longer the conversation, the bigger the savings — exactly when you need compaction most!

## Conclusion

In this cookbook, you learned how to manage long-running Claude conversations through session memory compaction.

### What We Covered

✅ **Effective compaction prompts** — Structured session memory that preserves user intent, completed work, errors, active work, and key references while discarding filler

✅ **Instant compaction** — Use background Promises (fire-and-forget) to proactively build session memory, eliminating user wait time when context limits are reached

✅ **Prompt caching for cost savings** — Reduce background update costs by ~80% by reusing the conversation prefix cache

✅ **Traditional vs. instant patterns** — Understand when to use each approach based on your application needs

### TypeScript vs. Python: Key Difference

| Concept | Python | TypeScript |
|---------|--------|------------|
| Background execution | `threading.Thread(daemon=True)` | Fire-and-forget `Promise` (no `await`) |
| Thread safety | `threading.Lock()` | Single-threaded event loop — no locks needed |
| Shared state update | `with self._lock: ...` | Direct assignment after `await` completes |

### Next Steps

- **For agentic workflows**: See [Automatic Context Compaction](../tool_use/automatic-context-compaction.ipynb) for SDK-based automatic compaction with tool use
- **For production**: Persist session memory to disk (e.g. `session-memory.json`) rather than keeping it in memory
- **For optimisation**: Experiment with `minTokensToInit` and `minTokensBetweenUpdates` thresholds to balance cost vs. freshness